# Experiment 6 – Fine-Tuning Hyperparameters of a Deep Learning Model

### Deep Learning Laboratory

**Aim:** To improve the performance of a deep learning model by fine-tuning its hyperparameters using TensorFlow/Keras.

### Learning Objectives
- Understand what hyperparameters are.
- Identify important hyperparameters in a neural network.
- Train a baseline model.
- Change selected hyperparameters and compare results.
- Select a better configuration based on validation accuracy.


## 1. Theory

**Hyperparameters** are settings chosen before training a neural network. They control how the model learns.

Common hyperparameters include:

| Hyperparameter | Meaning |
|---|---|
| Learning rate | Controls how large each weight update is |
| Batch size | Number of training samples processed at one time |
| Number of epochs | Number of times the model sees the training data |
| Number of neurons | Number of neurons in a layer |
| Number of hidden layers | Depth of the neural network |
| Optimizer | Method used to update model weights |
| Dropout rate | Fraction of neurons temporarily ignored during training |

### Experiment Idea

First, train a **baseline model**. Then create different configurations by changing the learning rate, batch size, hidden neurons and dropout. Finally, compare their validation accuracy.

```text
Dataset
   ↓
Baseline Model
   ↓
Change Hyperparameters
   ↓
Train Models
   ↓
Compare Validation Accuracy
   ↓
Select Better Model
```

In [ ]:
# Step 1: Import libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

In [ ]:
# Step 2: Load the MNIST handwritten digit dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize pixel values to 0–1
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Training images:", x_train.shape)
print("Testing images:", x_test.shape)

In [ ]:
# Step 3: Display sample images
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 2. Create a Function to Build the Model

The function below allows us to change important hyperparameters without rewriting the complete model.

We use a simple neural network for MNIST:

`Input Image → Flatten → Dense → Dropout → Output`

In [ ]:
# Step 4: Function to create a configurable neural network

def build_model(learning_rate=0.001, hidden_units=128, dropout_rate=0.2):
    model = keras.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(hidden_units, activation="relu"),
        layers.Dropout(dropout_rate),
        layers.Dense(10, activation="softmax")
    ])

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

print("Model-building function created successfully.")

## 3. Baseline Model

The baseline configuration is:

- Learning rate = 0.001
- Hidden neurons = 128
- Dropout = 0.20
- Batch size = 64
- Epochs = 5


In [ ]:
# Step 5: Train the baseline model
baseline_model = build_model(
    learning_rate=0.001,
    hidden_units=128,
    dropout_rate=0.20
)

baseline_history = baseline_model.fit(
    x_train,
    y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=64,
    verbose=1
)

baseline_loss, baseline_accuracy = baseline_model.evaluate(x_test, y_test, verbose=0)

print("\nBaseline Test Accuracy:", round(float(baseline_accuracy * 100), 2), "%")

## 4. Hyperparameter Experiment 1 – Learning Rate

Now we reduce the learning rate from `0.001` to `0.0005` and observe the change in performance.

In [ ]:
# Step 6: Model with a smaller learning rate
lr_model = build_model(
    learning_rate=0.0005,
    hidden_units=128,
    dropout_rate=0.20
)

lr_history = lr_model.fit(
    x_train,
    y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=64,
    verbose=0
)

lr_loss, lr_accuracy = lr_model.evaluate(x_test, y_test, verbose=0)
print("Learning-rate experiment accuracy:", round(float(lr_accuracy * 100), 2), "%")

## 5. Hyperparameter Experiment 2 – Number of Hidden Neurons

Change the number of neurons from `128` to `256`.

In [ ]:
# Step 7: Model with more hidden neurons
units_model = build_model(
    learning_rate=0.001,
    hidden_units=256,
    dropout_rate=0.20
)

units_history = units_model.fit(
    x_train,
    y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=64,
    verbose=0
)

units_loss, units_accuracy = units_model.evaluate(x_test, y_test, verbose=0)
print("Hidden-neuron experiment accuracy:", round(float(units_accuracy * 100), 2), "%")

## 6. Hyperparameter Experiment 3 – Dropout

Change the dropout rate from `0.20` to `0.40`. Dropout can help reduce overfitting by temporarily ignoring some neurons during training.

In [ ]:
# Step 8: Model with higher dropout
dropout_model = build_model(
    learning_rate=0.001,
    hidden_units=128,
    dropout_rate=0.40
)

dropout_history = dropout_model.fit(
    x_train,
    y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=64,
    verbose=0
)

dropout_loss, dropout_accuracy = dropout_model.evaluate(x_test, y_test, verbose=0)
print("Dropout experiment accuracy:", round(float(dropout_accuracy * 100), 2), "%")

## 7. Compare the Models

The table below compares the test accuracy obtained using different hyperparameter settings.

In [ ]:
# Step 9: Compare model performance
results = {
    "Baseline": baseline_accuracy * 100,
    "Learning Rate 0.0005": lr_accuracy * 100,
    "Hidden Units 256": units_accuracy * 100,
    "Dropout 0.40": dropout_accuracy * 100
}

for name, accuracy in results.items():
    print(f"{name:25s}: {accuracy:.2f}%")

best_model_name = max(results, key=results.get)
print("\nBest configuration:", best_model_name)
print("Best test accuracy:", f"{results[best_model_name]:.2f}%")

In [ ]:
# Step 10: Visual comparison of test accuracy
plt.figure(figsize=(9, 5))
plt.bar(results.keys(), results.values())
plt.ylabel("Test Accuracy (%)")
plt.xlabel("Model Configuration")
plt.title("Comparison of Hyperparameter Configurations")
plt.xticks(rotation=20)
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
# Step 11: Compare training/validation accuracy of baseline and best experiment
plt.figure(figsize=(9, 5))
plt.plot(baseline_history.history["val_accuracy"], label="Baseline Validation Accuracy")
plt.plot(lr_history.history["val_accuracy"], label="Learning Rate Validation Accuracy")
plt.plot(units_history.history["val_accuracy"], label="Hidden Units Validation Accuracy")
plt.plot(dropout_history.history["val_accuracy"], label="Dropout Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy Comparison")
plt.legend()
plt.show()

## 8. Student Practice

Try your own configuration:

- Learning rate = `0.01` or `0.0001`
- Hidden units = `64`, `128`, or `256`
- Dropout = `0.1`, `0.2`, or `0.5`
- Batch size = `32`, `64`, or `128`
- Epochs = `5` or `10`

Record the test accuracy and compare it with the baseline model.

In [ ]:
# Step 12: Student-defined hyperparameters
student_learning_rate = 0.001
student_hidden_units = 256
student_dropout = 0.3
student_batch_size = 32
student_epochs = 5

student_model = build_model(
    learning_rate=student_learning_rate,
    hidden_units=student_hidden_units,
    dropout_rate=student_dropout
)

student_model.fit(
    x_train,
    y_train,
    validation_split=0.1,
    epochs=student_epochs,
    batch_size=student_batch_size,
    verbose=0
)

student_loss, student_accuracy = student_model.evaluate(x_test, y_test, verbose=0)
print("Student model test accuracy:", round(float(student_accuracy * 100), 2), "%")

## Result

Thus, a deep learning model was trained and its performance was improved and compared by fine-tuning different hyperparameters such as learning rate, number of hidden neurons, dropout rate and batch size.

## Viva Questions

1. What is a hyperparameter?
2. Give four examples of hyperparameters.
3. What is learning rate?
4. What happens if the learning rate is too high?
5. What is batch size?
6. What is an epoch?
7. What is dropout?
8. Why do we tune hyperparameters?
9. What is overfitting?
10. How do you select the best hyperparameter configuration?